# GeoCrop — web API from Jupyter

Start the local server separately with `pip install -e ".[web]"` and `geocrop serve --workspace /path/to/study`. Set the same random `GEOCROP_WEB_TOKEN` in server and kernel environments; do not save it in cells or outputs. Command paths refer to the server workspace. This notebook performs software checks only.


In [ ]:
import json
import os
import tempfile
import uuid
from pathlib import Path

from geocrop.client import EngineClient

engine = EngineClient(base_url=os.environ.get("GEOCROP_WEB_URL", "http://127.0.0.1:8765"))
engine.health()


## Submit an existing engine command and wait for its outcome


In [ ]:
job = engine.submit(["indices", "--format", "json"])
result = engine.wait(job["id"], timeout=60)
indices = json.loads(engine.logs(job["id"])["text"])
indices[:2]


## Produce and download a file

A fresh output directory avoids replacing existing study results. The generated manifest is software/provenance metadata, not an observational study result.


In [ ]:
output = "outputs/notebook-" + uuid.uuid4().hex[:12]
job = engine.submit(["manifest", "--out", output])
engine.wait(job["id"], timeout=60)
files = engine.artifacts(job["id"])
files


In [ ]:
manifest = next(f for f in files if f["path"].endswith("run_manifest.json"))
destination = Path(tempfile.mkdtemp(prefix="geocrop-notebook-")) / "run_manifest.json"
engine.download(job["id"], manifest["path"], destination)
json.loads(destination.read_text())["schema"]


## More workflows

Use `engine.commands()` to inspect available commands. Submit `run --config study.yaml`, `train`, `predict`, `parcels` or `benchmark` using the same CLI arguments as the terminal. A full `demo --quick` uses synthetic data; it does not measure real-world accuracy.

Use `engine.wait(job_id, check=False)` to inspect non-success outcomes, `engine.logs(job_id, offset=...)` for incremental logs and `engine.cancel(job_id)` to stop a job. Timeouts do not cancel jobs.


In [ ]:
engine.close()
